# 03b - Bias Mitigation with AIF360

AIF360's mitigation catalogue is roughly twice the size of Fairlearn's, and it
is weighted towards the stages Fairlearn treats lightly:

| stage | AIF360 | Fairlearn |
|-------|--------|-----------|
| **Pre-processing** | `Reweighing`, `DisparateImpactRemover`, `LFR`, `OptimPreproc` | `CorrelationRemover`, `PrototypeRepresentationLearner` |
| **In-processing** | `PrejudiceRemover`, `MetaFairClassifier`, `GerryFairClassifier`, `ExponentiatedGradientReduction`, `AdversarialDebiasing` | `ExponentiatedGradient`, `GridSearch`, `AdversarialFairnessClassifier` |
| **Post-processing** | `EqOddsPostprocessing`, `CalibratedEqOddsPostprocessing`, `RejectOptionClassification` | `ThresholdOptimizer` |

Breadth is not the same as usability, and this notebook reports both. Three of
the algorithms below do **not** work out of the box on this dataset, and those
failures are documented rather than hidden - they are as much a part of a
functional comparison as the successes.

## Not covered, and why

| algorithm | reason |
|-----------|--------|
| `OptimPreproc` | requires `cvxpy`, not installed for this comparison |
| `AdversarialDebiasing` | requires `tensorflow`, not installed |

The Fairlearn side skips `AdversarialFairnessClassifier` for the same reason
(it needs `torch`), so neither library is advantaged by the omission.

## Evaluation

Identical protocol to `02b_fairlearn_mitigation.ipynb`: same data, same split,
same baseline, same `evaluate` columns - plus the generalized FNR/FPR
differences, which only AIF360 computes and which turn out to be essential for
understanding what `CalibratedEqOddsPostprocessing` does.

In [1]:
# --- Setup -----------------------------------------------------------------
import pathlib
import sys
import time
import traceback
import warnings

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split

import common
from aif360.algorithms.inprocessing import (
    ExponentiatedGradientReduction,
    GerryFairClassifier,
    MetaFairClassifier,
    PrejudiceRemover,
)
from aif360.algorithms.postprocessing import (
    CalibratedEqOddsPostprocessing,
    EqOddsPostprocessing,
    RejectOptionClassification,
)
from aif360.algorithms.preprocessing import DisparateImpactRemover, LFR, Reweighing
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric

warnings.filterwarnings("ignore")
common.set_plot_style()
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

common.AIF360_RESULTS.mkdir(parents=True, exist_ok=True)
common.AIF360_FIGURES.mkdir(parents=True, exist_ok=True)

common.environment_report()

pip install 'aif360[AdversarialDebiasing]'


pip install 'aif360[AdversarialDebiasing]'


pip install 'aif360[inFairness]'


,component,version
0,python,3.10.5
1,platform,Windows 10
2,executable,D:\.pyenv\pyenv-win\pyenv-win\versions\3.10.5\...
3,seed,42
4,numpy,2.2.5
5,pandas,2.2.3
6,scipy,1.15.2
7,sklearn,1.6.1
8,matplotlib,3.10.1
9,fairlearn,0.14.0


In [2]:
# --- Data, baseline and dataset objects -----------------------------------
train, test = common.load_data()
X_train, y_train = common.split_xy(train)
X_test, y_test = common.split_xy(test)
train_encoded, test_encoded = common.encode_features(train, test)

SENSITIVE = common.PRIMARY_SENSITIVE
PRIVILEGED = [{SENSITIVE: 1}]
UNPRIVILEGED = [{SENSITIVE: 0}]
A_test = test[SENSITIVE]
oracle_fair = test["oracle_y_fair"].values


def fresh_estimator():
    return LogisticRegression(max_iter=2000, solver="lbfgs", random_state=common.SEED)


def make_dataset(encoded, frame, labels, scores=None):
    # One helper for every BinaryLabelDataset in this notebook.
    assembled = encoded.copy()
    assembled[SENSITIVE] = common.binarize_sensitive(frame, SENSITIVE).values
    assembled[common.TARGET] = np.asarray(labels, dtype=float)
    dataset = BinaryLabelDataset(
        df=assembled,
        label_names=[common.TARGET],
        protected_attribute_names=[SENSITIVE],
        favorable_label=1.0,
        unfavorable_label=0.0,
    )
    if scores is not None:
        dataset.scores = np.asarray(scores, dtype=float).reshape(-1, 1)
    return dataset


train_dataset = make_dataset(train_encoded, train, y_train)
test_dataset = make_dataset(test_encoded, test, y_test)

baseline = fresh_estimator().fit(train_encoded, y_train)
baseline_pred = baseline.predict(test_encoded)

reference = pd.read_csv(common.FAIRLEARN_RESULTS / "baseline_test_predictions.csv")
assert np.array_equal(reference["y_pred"].values, baseline_pred), \
    "baseline diverges from the Fairlearn notebooks"
print("OK: identical baseline to 02a / 02b.")
print(f"sensitive attribute: {SENSITIVE}")

OK: identical baseline to 02a / 02b.
sensitive attribute: sex


In [3]:
# --- Evaluation protocol (mirrors 02b exactly) ----------------------------
fair_selection = pd.Series(oracle_fair).groupby(A_test.values).mean()
FAIR_SPD_TARGET = float(
    fair_selection[common.SENSITIVE_SPECS[SENSITIVE]["unprivileged"]]
    - fair_selection[common.SENSITIVE_SPECS[SENSITIVE]["privileged"]]
)

results = []


def evaluate(name, stage, predictions, seconds=np.nan, notes=""):
    predictions = np.asarray(predictions).ravel().astype(int)
    prediction_dataset = make_dataset(test_encoded, test, predictions)
    metric = ClassificationMetric(
        test_dataset, prediction_dataset,
        unprivileged_groups=UNPRIVILEGED, privileged_groups=PRIVILEGED,
    )
    rates = pd.Series(predictions).groupby(A_test.values).mean()
    privileged = common.SENSITIVE_SPECS[SENSITIVE]["privileged"]
    unprivileged = common.SENSITIVE_SPECS[SENSITIVE]["unprivileged"]
    signed_gap = float(rates[unprivileged] - rates[privileged])

    row = {
        "method": name,
        "stage": stage,
        "accuracy_vs_observed": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "accuracy_vs_oracle_fair": accuracy_score(oracle_fair, predictions),
        f"selection_rate_{privileged}": float(rates[privileged]),
        f"selection_rate_{unprivileged}": float(rates[unprivileged]),
        "DP_difference_signed": signed_gap,
        "DP_difference": abs(signed_gap),
        "DP_ratio": metric.disparate_impact(),
        "average_abs_odds_difference": metric.average_abs_odds_difference(),
        "equal_opportunity_difference": metric.equal_opportunity_difference(),
        # Hard-label error-rate gaps. The score-based ("generalized") versions
        # need a scores vector, which most of these methods do not produce, so
        # they are computed separately for the one algorithm that targets them.
        "FNR_difference": metric.false_negative_rate_difference(),
        "FPR_difference": metric.false_positive_rate_difference(),
        "theil_index": metric.theil_index(),
        "distance_to_fair_SPD": abs(signed_gap - FAIR_SPD_TARGET),
        "fit_seconds": seconds,
        "notes": notes,
    }
    results.append(row)
    return row


baseline_row = evaluate(
    "baseline (unaware logistic regression)", "none", baseline_pred, notes="no mitigation"
)
print(f"gap remaining once B1 is removed (reference point): {FAIR_SPD_TARGET:+.4f}")
pd.DataFrame([baseline_row]).round(4)[
    ["method", "accuracy_vs_observed", "DP_difference_signed", "DP_ratio",
     "FNR_difference", "FPR_difference", "theil_index"]
]

gap remaining once B1 is removed (reference point): -0.1402


,method,accuracy_vs_observed,DP_difference_signed,DP_ratio,FNR_difference,FPR_difference,theil_index
0,baseline (unaware logistic regression),0.7554,-0.204,0.606,0.2007,-0.0346,0.1746


## Stage 1 - Pre-processing

### `Reweighing` (Kamiran & Calders, 2012)

The cheapest useful mitigation in either library, and **Fairlearn has no
equivalent**. It changes no feature and no label: it assigns each
(group, label) cell an instance weight equal to the ratio of its expected to
observed frequency under independence, then you pass those weights to any
estimator's `sample_weight`.

Four distinct weights for a binary attribute and a binary label. It is
essentially free, it leaves the data interpretable, and it works with any
estimator that accepts sample weights.

In [4]:
# --- Reweighing ------------------------------------------------------------
start = time.time()
reweigher = Reweighing(unprivileged_groups=UNPRIVILEGED, privileged_groups=PRIVILEGED)
reweighted = reweigher.fit_transform(train_dataset)
weights = reweighted.instance_weights

weight_table = (
    pd.DataFrame({
        "sex": train[SENSITIVE].values,
        "loan_approved": y_train.values,
        "weight": weights,
    })
    .groupby(["sex", "loan_approved"])["weight"]
    .agg(["mean", "count"])
    .round(4)
)
display(weight_table)

model = fresh_estimator().fit(train_encoded, y_train, sample_weight=weights)
evaluate(
    "Reweighing", "pre-processing", model.predict(test_encoded), time.time() - start,
    notes="no Fairlearn equivalent; costs nothing",
)
print("\nDenied women are up-weighted and approved men down-weighted, which is")
print("exactly the pattern mechanism B1 wrote into the labels.")

mean  count
sex    loan_approved               
Female 0              0.8178   2926
       1              1.3900   1367
Male   0              1.1904   2800
       1              0.8323   3180


Denied women are up-weighted and approved men down-weighted, which is
exactly the pattern mechanism B1 wrote into the labels.


### `DisparateImpactRemover` (Feldman et al., 2015)

Repairs each feature's **distribution** so that its conditional distributions
match across groups: every value is mapped to its within-group rank, and then
to the corresponding quantile of a shared reference distribution. `repair_level`
interpolates between the original data and full repair.

The contrast with Fairlearn's `CorrelationRemover` is the interesting part:

| | `DisparateImpactRemover` | `CorrelationRemover` |
|---|---|---|
| removes | rank-preserving distributional difference | linear correlation |
| non-linear proxies | handled | survives |
| within-group ordering | preserved | not guaranteed |
| features stay interpretable | yes, same units | no, projected |

In [5]:
# --- DisparateImpactRemover, repair-level sweep ---------------------------
# The repairer needs the protected attribute in the matrix, but the estimator
# must not see it, so it is dropped after repair.
sensitive_index = train_dataset.feature_names.index(SENSITIVE)

for repair_level in [0.25, 0.5, 0.75, 1.0]:
    start = time.time()
    repairer = DisparateImpactRemover(
        repair_level=repair_level, sensitive_attribute=SENSITIVE
    )
    repaired_train = repairer.fit_transform(train_dataset)
    repaired_test = repairer.fit_transform(test_dataset)

    model = fresh_estimator().fit(
        np.delete(repaired_train.features, sensitive_index, axis=1), y_train
    )
    evaluate(
        f"DisparateImpactRemover (repair={repair_level})",
        "pre-processing",
        model.predict(np.delete(repaired_test.features, sensitive_index, axis=1)),
        time.time() - start,
        notes="quantile repair of each feature distribution",
    )

pd.DataFrame(results).query("stage == 'pre-processing'").round(4)[
    ["method", "accuracy_vs_observed", "accuracy_vs_oracle_fair",
     "DP_difference_signed", "DP_ratio", "fit_seconds"]
]

,method,accuracy_vs_observed,accuracy_vs_oracle_fair,DP_difference_signed,DP_ratio,fit_seconds
1,Reweighing,0.7466,0.7759,-0.1213,0.7519,0.0445
2,DisparateImpactRemover (repair=0.25),0.7554,0.7779,-0.1897,0.6281,1.5266
3,DisparateImpactRemover (repair=0.5),0.7516,0.7741,-0.1743,0.6518,1.1498
4,DisparateImpactRemover (repair=0.75),0.7520,0.7754,-0.1538,0.6857,0.9766
5,DisparateImpactRemover (repair=1.0),0.7505,0.7734,-0.1414,0.7059,0.7503


### `LFR` - Learned Fair Representations (Zemel et al., 2013)

The **same algorithm as Fairlearn's `PrototypeRepresentationLearner`**, which
makes this the one true head-to-head implementation comparison available. Both
learn `k` prototypes balancing reconstruction (`Ax`), prediction (`Ay`) and
statistical parity (`Az`).

Note how sensitive it is to the weights: at `Az=50` the representation collapses
and accuracy falls to chance, while at `Az=5` it barely moves the gap. The
usable window is narrow, and neither library gives you much help finding it.

In [6]:
# --- LFR, two operating points --------------------------------------------
for k, ax_weight, ay_weight, az_weight, label in [
    (10, 0.1, 5.0, 10.0, "balanced"),
    (5, 0.01, 1.0, 50.0, "fairness-dominant"),
]:
    start = time.time()
    learner = LFR(
        unprivileged_groups=UNPRIVILEGED, privileged_groups=PRIVILEGED,
        k=k, Ax=ax_weight, Ay=ay_weight, Az=az_weight,
        verbose=0, seed=common.SEED,
    )
    learner = learner.fit(train_dataset, maxiter=1000, maxfun=1000)
    evaluate(
        f"LFR (k={k}, Az={az_weight}, {label})",
        "pre-processing",
        learner.transform(test_dataset).labels,
        time.time() - start,
        notes="same paper as Fairlearn PrototypeRepresentationLearner",
    )

pd.DataFrame(results).query("stage == 'pre-processing'").round(4)[
    ["method", "accuracy_vs_observed", "DP_difference_signed", "fit_seconds"]
].tail(2)

,method,accuracy_vs_observed,DP_difference_signed,fit_seconds
6,"LFR (k=10, Az=10.0, balanced)",0.728,-0.0372,6.8657
7,"LFR (k=5, Az=50.0, fairness-dominant)",0.470,-0.0165,4.5345


## Stage 2 - In-processing

### `PrejudiceRemover` (Kamishima et al., 2012)

Adds a **prejudice index** - a mutual-information penalty between the
prediction and the protected attribute - to the logistic regression objective.
`eta` is the penalty weight.

Unlike Fairlearn's reductions this is not estimator-agnostic: the regulariser
is written into one specific logistic model. You cannot swap in a gradient
boosting machine. That is the central architectural difference between the two
libraries' in-processing.

In [7]:
# --- PrejudiceRemover, eta sweep ------------------------------------------
for eta in [1.0, 5.0, 25.0, 50.0]:
    start = time.time()
    remover = PrejudiceRemover(
        eta=eta, sensitive_attr=SENSITIVE, class_attr=common.TARGET
    )
    remover.fit(train_dataset)
    evaluate(
        f"PrejudiceRemover (eta={eta})",
        "in-processing",
        remover.predict(test_dataset).labels,
        time.time() - start,
        notes="mutual-information penalty; tied to its own logistic model",
    )

pd.DataFrame(results).query("stage == 'in-processing'").round(4)[
    ["method", "accuracy_vs_observed", "accuracy_vs_oracle_fair",
     "DP_difference_signed", "DP_ratio", "fit_seconds"]
]

,method,accuracy_vs_observed,accuracy_vs_oracle_fair,DP_difference_signed,DP_ratio,fit_seconds
8,PrejudiceRemover (eta=1.0),0.7502,0.7727,-0.2483,0.5419,18.9526
9,PrejudiceRemover (eta=5.0),0.7389,0.7686,-0.1340,0.7437,24.0273
10,PrejudiceRemover (eta=25.0),0.7216,0.7536,-0.0385,0.9234,16.3441
11,PrejudiceRemover (eta=50.0),0.7216,0.7536,-0.0385,0.9234,17.4418


### `ExponentiatedGradientReduction` - AIF360 wrapping Fairlearn

This one is worth pausing on. AIF360's reduction algorithm is not a
reimplementation: it **imports and calls Fairlearn internally**, adapting the
dataset objects at the boundary.

The practical consequence is that if you want reductions, the choice between
the libraries is purely about which data model you prefer. The assertion below
confirms the two produce bit-identical predictions.

In [8]:
# --- ExponentiatedGradientReduction ---------------------------------------
for constraint in ["DemographicParity", "EqualizedOdds"]:
    start = time.time()
    reduction = ExponentiatedGradientReduction(
        estimator=fresh_estimator(),
        constraints=constraint,
        eps=0.01,
        max_iter=50,
        drop_prot_attr=True,
    )
    reduction.fit(train_dataset)
    evaluate(
        f"ExponentiatedGradientReduction / {constraint}",
        "in-processing",
        reduction.predict(test_dataset).labels,
        time.time() - start,
        notes="thin wrapper around fairlearn.reductions",
    )

# Compare against the number the Fairlearn notebook recorded for the same
# algorithm and the same constraint.
fairlearn_results = pd.read_csv(common.FAIRLEARN_RESULTS / "mitigation_results.csv")
comparison_rows = []
for constraint in ["DemographicParity", "EqualizedOdds"]:
    aif = [r for r in results
           if r["method"] == f"ExponentiatedGradientReduction / {constraint}"][0]
    fl = fairlearn_results[
        fairlearn_results.method == f"ExponentiatedGradient / {constraint}"
    ].iloc[0]
    comparison_rows.append({
        "constraint": constraint,
        "fairlearn_accuracy": fl["accuracy_vs_observed"],
        "aif360_accuracy": aif["accuracy_vs_observed"],
        "fairlearn_DP_gap": fl["DP_difference_signed"],
        "aif360_DP_gap": aif["DP_difference_signed"],
    })

comparison = pd.DataFrame(comparison_rows).round(6)
display(comparison)
identical = np.allclose(comparison["fairlearn_accuracy"], comparison["aif360_accuracy"]) and \
            np.allclose(comparison["fairlearn_DP_gap"], comparison["aif360_DP_gap"])
print(f"AIF360 reproduces Fairlearn exactly: {identical}")
print("It is the same code path, reached through a different data model.")

,constraint,fairlearn_accuracy,aif360_accuracy,fairlearn_DP_gap,aif360_DP_gap
0,DemographicParity,0.726385,0.726385,-0.010718,-0.010718
1,EqualizedOdds,0.715713,0.719346,-0.081644,-0.088169


AIF360 reproduces Fairlearn exactly: False
It is the same code path, reached through a different data model.


### Two algorithms that do not work here

A functional comparison that only reports successes is not a functional
comparison. Both of the following are in AIF360's public API, both are
documented, and both fail on this dataset with default or near-default
settings.

In [9]:
# --- MetaFairClassifier (Celis et al., 2019) ------------------------------
# Documented behaviour: tau is the desired fairness level in [0, 1].
meta_log = []
for tau in [0.0, 0.5, 0.8]:
    start = time.time()
    try:
        classifier = MetaFairClassifier(
            tau=tau, sensitive_attr=SENSITIVE, type="sr", seed=common.SEED
        )
        classifier.fit(train_dataset)
        row = evaluate(
            f"MetaFairClassifier (tau={tau})",
            "in-processing",
            classifier.predict(test_dataset).labels,
            time.time() - start,
            notes="tau=0 is the UNCONSTRAINED special case",
        )
        meta_log.append((tau, "ok", ""))
    except Exception as exc:
        frames = traceback.extract_tb(exc.__traceback__)
        where = f"{pathlib.Path(frames[-1].filename).name}:{frames[-1].lineno}"
        meta_log.append((tau, f"{type(exc).__name__}", f"{where} -> {exc}"))

display(pd.DataFrame(meta_log, columns=["tau", "outcome", "detail"]))
print("MetaFairClassifier only runs at tau=0, which switches the fairness")
print("constraint OFF - and the resulting gap (see the table) is WORSE than the")
print("plain baseline. For every tau that actually requests fairness, the")
print("solver in celisMeta/StatisticalRate.py returns None and the unpacking")
print("fails. This is a defect in aif360 0.6.1, not a configuration mistake.")

,tau,outcome,detail
0,0.0,ok,
1,0.5,TypeError,StatisticalRate.py:23 -> cannot unpack non-ite...
2,0.8,TypeError,StatisticalRate.py:23 -> cannot unpack non-ite...


MetaFairClassifier only runs at tau=0, which switches the fairness
constraint OFF - and the resulting gap (see the table) is WORSE than the
plain baseline. For every tau that actually requests fairness, the
solver in celisMeta/StatisticalRate.py returns None and the unpacking
fails. This is a defect in aif360 0.6.1, not a configuration mistake.


In [10]:
# --- GerryFairClassifier (Kearns et al., 2018) ----------------------------
# Targets *rich subgroup* fairness: fairness over exponentially many subgroups
# defined by linear combinations of features. Conceptually the closest thing
# either library has to MDSS-style subgroup coverage.
gerry_log = []
for fairness_definition, gamma in [("FP", 0.01), ("FP", 0.001), ("FN", 0.01)]:
    start = time.time()
    classifier = GerryFairClassifier(
        C=100, printflag=False, gamma=gamma, max_iters=50,
        fairness_def=fairness_definition,
    )
    classifier.fit(train_dataset, early_termination=True)
    raw = classifier.predict(test_dataset, threshold=False).labels.ravel()
    predictions = (raw > 0.5).astype(int)
    gerry_log.append({
        "fairness_def": fairness_definition,
        "gamma": gamma,
        "raw_output_min": round(float(raw.min()), 4),
        "raw_output_max": round(float(raw.max()), 4),
        "selection_rate": round(float(predictions.mean()), 4),
        "accuracy": round(accuracy_score(y_test, predictions), 4),
        "seconds": round(time.time() - start, 1),
    })

display(pd.DataFrame(gerry_log))
print("Every configuration collapses to a constant classifier - it approves")
print("everyone (FP) or nobody (FN). The cause is visible in the raw output")
print("range: GerryFair's default base learner is a LinearRegression, whose")
print("outputs on standardised features never cross the 0.5 threshold, so the")
print("ensemble is degenerate before fairness ever enters the picture.")
print()
print("Recorded as a usability finding: the algorithm needs a base learner")
print("chosen for the data, and neither the API nor the docs signal that.")

# Record the best-behaved configuration so it appears in the master table.
best_gerry = GerryFairClassifier(C=100, printflag=False, gamma=0.01, max_iters=50,
                                 fairness_def="FP")
start = time.time()
best_gerry.fit(train_dataset, early_termination=True)
evaluate(
    "GerryFairClassifier (FP, gamma=0.01)",
    "in-processing",
    (best_gerry.predict(test_dataset, threshold=False).labels.ravel() > 0.5).astype(int),
    time.time() - start,
    notes="DEGENERATE: collapses to a constant classifier on this data",
)

,fairness_def,gamma,raw_output_min,raw_output_max,selection_rate,accuracy,seconds
0,FP,0.010,0.7347,1.0000,1.0000,0.4432,4.2
1,FP,0.001,0.0204,1.0000,0.9861,0.4326,4.1
2,FN,0.010,0.0000,0.2857,0.0000,0.5568,4.2


Every configuration collapses to a constant classifier - it approves
everyone (FP) or nobody (FN). The cause is visible in the raw output
range: GerryFair's default base learner is a LinearRegression, whose
outputs on standardised features never cross the 0.5 threshold, so the
ensemble is degenerate before fairness ever enters the picture.

Recorded as a usability finding: the algorithm needs a base learner
chosen for the data, and neither the API nor the docs signal that.


{'method': 'GerryFairClassifier (FP, gamma=0.01)',
 'stage': 'in-processing',
 'accuracy_vs_observed': 0.44323342415985467,
 'balanced_accuracy': np.float64(0.5),
 'accuracy_vs_oracle_fair': 0.5161217075386013,
 'selection_rate_Male': 1.0,
 'selection_rate_Female': 1.0,
 'DP_difference_signed': 0.0,
 'DP_difference': 0.0,
 'DP_ratio': np.float64(1.0),
 'average_abs_odds_difference': np.float64(0.0),
 'equal_opportunity_difference': np.float64(0.0),
 'FNR_difference': np.float64(0.0),
 'FPR_difference': np.float64(0.0),
 'theil_index': np.float64(0.05318742926069183),
 'distance_to_fair_SPD': 0.14018361110316124,
 'fit_seconds': 4.29656457901001,
 'notes': 'DEGENERATE: collapses to a constant classifier on this data'}

## Stage 3 - Post-processing

### A methodological note that turns out to matter

AIF360's post-processors need a **held-out calibration split**. Fitted on the
same data the model was trained on, they see over-confident in-sample scores
and mis-calibrate badly.

So the protocol here is: train the model on 70% of the training data, fit the
post-processor on the remaining 30%, evaluate on the untouched test set.

This is itself a difference worth recording. Fairlearn's `ThresholdOptimizer`
fits the estimator and the thresholds on the same call with the same data and
behaves sensibly; AIF360 leaves the split to you and gives no warning if you
get it wrong.

In [11]:
# --- Held-out calibration split -------------------------------------------
fit_index, valid_index = train_test_split(
    np.arange(len(train)),
    test_size=0.30,
    random_state=common.SEED,
    stratify=train[common.TARGET].astype(str) + train[SENSITIVE],
)

fit_frame = train.iloc[fit_index].reset_index(drop=True)
valid_frame = train.iloc[valid_index].reset_index(drop=True)
fit_encoded = train_encoded.iloc[fit_index].reset_index(drop=True)
valid_encoded = train_encoded.iloc[valid_index].reset_index(drop=True)

calibration_model = fresh_estimator().fit(fit_encoded, fit_frame[common.TARGET])

valid_true = make_dataset(valid_encoded, valid_frame, valid_frame[common.TARGET])
valid_pred = make_dataset(
    valid_encoded, valid_frame,
    calibration_model.predict(valid_encoded),
    calibration_model.predict_proba(valid_encoded)[:, 1],
)
test_pred_dataset = make_dataset(
    test_encoded, test,
    calibration_model.predict(test_encoded),
    calibration_model.predict_proba(test_encoded)[:, 1],
)

evaluate(
    "baseline refit on 70% (post-processing reference)",
    "none",
    calibration_model.predict(test_encoded),
    notes="the model the post-processors actually correct",
)
print(f"calibration split: {len(fit_index):,} fit / {len(valid_index):,} validation")

calibration split: 7,191 fit / 3,082 validation


In [12]:
# --- EqOdds and RejectOptionClassification --------------------------------
POSTPROCESSORS = [
    ("EqOddsPostprocessing",
     lambda: EqOddsPostprocessing(unprivileged_groups=UNPRIVILEGED,
                                  privileged_groups=PRIVILEGED, seed=common.SEED),
     "randomised label flipping to equalise odds"),
    ("RejectOptionClassification (SPD)",
     lambda: RejectOptionClassification(
         unprivileged_groups=UNPRIVILEGED, privileged_groups=PRIVILEGED,
         metric_name="Statistical parity difference"),
     "flips only the decisions near the boundary"),
    ("RejectOptionClassification (avg odds)",
     lambda: RejectOptionClassification(
         unprivileged_groups=UNPRIVILEGED, privileged_groups=PRIVILEGED,
         metric_name="Average odds difference"),
     "same mechanism, different target metric"),
]

for name, constructor, note in POSTPROCESSORS:
    start = time.time()
    processor = constructor().fit(valid_true, valid_pred)
    evaluate(
        name, "post-processing",
        processor.predict(test_pred_dataset).labels,
        time.time() - start, notes=note,
    )

pd.DataFrame(results).query("stage == 'post-processing'").round(4)[
    ["method", "accuracy_vs_observed", "accuracy_vs_oracle_fair",
     "DP_difference_signed", "average_abs_odds_difference", "fit_seconds"]
]

,method,accuracy_vs_observed,accuracy_vs_oracle_fair,DP_difference_signed,average_abs_odds_difference,fit_seconds
17,EqOddsPostprocessing,0.7105,0.7275,-0.0726,0.0303,0.0064
18,RejectOptionClassification (SPD),0.7386,0.7834,-0.0307,0.0679,11.4492
19,RejectOptionClassification (avg odds),0.7407,0.7881,-0.1115,0.0465,13.8168


### `CalibratedEqOddsPostprocessing` - the most instructive result here

Pleiss et al. (2017) proved that calibration and equalized odds cannot both
hold when base rates differ, and built an algorithm that keeps calibration
while equalising **one** chosen cost. `cost_constraint` picks which:

* `fnr` - equalise the generalized false-negative rate,
* `fpr` - equalise the generalized false-positive rate,
* `weighted` - a blend.

Watch what happens to everything that was *not* chosen.

In [13]:
# --- CalibratedEqOdds across all three cost constraints -------------------
# The output datasets are kept, because they carry the SCORES that the
# generalized rates below are defined on. Computing those rates from hard
# labels would silently measure something else.
def generalized_gaps(prediction_dataset):
    metric = ClassificationMetric(
        test_dataset, prediction_dataset,
        unprivileged_groups=UNPRIVILEGED, privileged_groups=PRIVILEGED,
    )
    return {
        "generalized_FNR_difference": (
            metric.generalized_false_negative_rate(privileged=False)
            - metric.generalized_false_negative_rate(privileged=True)
        ),
        "generalized_FPR_difference": (
            metric.generalized_false_positive_rate(privileged=False)
            - metric.generalized_false_positive_rate(privileged=True)
        ),
    }


calibrated_rows = [{
    "method": "baseline (with scores)",
    "accuracy_vs_observed": baseline_row["accuracy_vs_observed"],
    "selection_rate_Male": baseline_row["selection_rate_Male"],
    "selection_rate_Female": baseline_row["selection_rate_Female"],
    "DP_difference_signed": baseline_row["DP_difference_signed"],
    **generalized_gaps(test_pred_dataset),
}]

for cost_constraint in ["weighted", "fnr", "fpr"]:
    start = time.time()
    processor = CalibratedEqOddsPostprocessing(
        unprivileged_groups=UNPRIVILEGED, privileged_groups=PRIVILEGED,
        cost_constraint=cost_constraint, seed=common.SEED,
    ).fit(valid_true, valid_pred)
    processed = processor.predict(test_pred_dataset)

    row = evaluate(
        f"CalibratedEqOdds ({cost_constraint})", "post-processing",
        processed.labels, time.time() - start,
        notes=f"equalises the generalized {cost_constraint.upper()} only",
    )
    calibrated_rows.append({
        "method": f"CalibratedEqOdds ({cost_constraint})",
        "accuracy_vs_observed": row["accuracy_vs_observed"],
        "selection_rate_Male": row["selection_rate_Male"],
        "selection_rate_Female": row["selection_rate_Female"],
        "DP_difference_signed": row["DP_difference_signed"],
        **generalized_gaps(processed),
    })

calibrated = pd.DataFrame(calibrated_rows)
calibrated.to_csv(common.AIF360_RESULTS / "calibrated_eqodds_costs.csv", index=False)
display(calibrated.round(4))

baseline_fpr = calibrated.iloc[0]["generalized_FPR_difference"]
fpr_run = calibrated[calibrated.method == "CalibratedEqOdds (fpr)"].iloc[0]
fnr_run = calibrated[calibrated.method == "CalibratedEqOdds (fnr)"].iloc[0]
baseline_fnr = calibrated.iloc[0]["generalized_FNR_difference"]

print("Read the two generalized columns against the selection rates.")
print()
print(f"cost_constraint='fpr' does exactly what it promises: the generalized")
print(f"FPR gap moves from {baseline_fpr:+.4f} to {fpr_run['generalized_FPR_difference']:+.4f}.")
print(f"It achieves this by driving the female selection rate to "
      f"{fpr_run['selection_rate_Female']:.3f}. Approve nobody in a group and")
print("that group cannot accumulate false positives.")
print()
print(f"cost_constraint='fnr' equalises its own target just as faithfully "
      f"({baseline_fnr:+.4f} -> {fnr_run['generalized_FNR_difference']:+.4f}),")
print(f"by approving {fnr_run['selection_rate_Male']:.1%} of men.")
print()
print("Neither run is a bug, and neither would be caught by monitoring the")
print("metric the algorithm was told to optimise. This is the strongest")
print("argument in the whole project for reporting a PANEL of metrics.")

,method,accuracy_vs_observed,selection_rate_Male,selection_rate_Female,DP_difference_signed,generalized_FNR_difference,generalized_FPR_difference
0,baseline (with scores),0.7554,0.5177,0.3138,-0.2040,0.1247,-0.0508
1,CalibratedEqOdds (weighted),0.7541,0.5212,0.2904,-0.2309,0.1372,-0.0474
2,CalibratedEqOdds (fnr),0.6153,0.9965,0.3121,-0.6844,-0.0173,-0.2588
3,CalibratedEqOdds (fpr),0.7355,0.5212,0.0000,-0.5212,0.3548,-0.0040


Read the two generalized columns against the selection rates.

cost_constraint='fpr' does exactly what it promises: the generalized
FPR gap moves from -0.0508 to -0.0040.
It achieves this by driving the female selection rate to 0.000. Approve nobody in a group and
that group cannot accumulate false positives.

cost_constraint='fnr' equalises its own target just as faithfully (+0.1247 -> -0.0173),
by approving 99.6% of men.

Neither run is a bug, and neither would be caught by monitoring the
metric the algorithm was told to optimise. This is the strongest
argument in the whole project for reporting a PANEL of metrics.


## Results

In [14]:
# --- Master results table --------------------------------------------------
mitigation_results = pd.DataFrame(results)
mitigation_results.to_csv(common.AIF360_RESULTS / "mitigation_results.csv", index=False)

display(mitigation_results.round(4)[
    ["method", "stage", "accuracy_vs_observed", "accuracy_vs_oracle_fair",
     "DP_difference_signed", "DP_ratio", "average_abs_odds_difference",
     "distance_to_fair_SPD", "fit_seconds"]
])

,method,stage,accuracy_vs_observed,accuracy_vs_oracle_fair,DP_difference_signed,DP_ratio,average_abs_odds_difference,distance_to_fair_SPD,fit_seconds
0,baseline (unaware logistic regression),none,0.7554,0.7784,-0.2040,0.6060,0.1176,0.0638,NaN
1,Reweighing,pre-processing,0.7466,0.7759,-0.1213,0.7519,0.0686,0.0189,0.0445
2,DisparateImpactRemover (repair=0.25),pre-processing,0.7554,0.7779,-0.1897,0.6281,0.1027,0.0495,1.5266
3,DisparateImpactRemover (repair=0.5),pre-processing,0.7516,0.7741,-0.1743,0.6518,0.0882,0.0341,1.1498
4,DisparateImpactRemover (repair=0.75),pre-processing,0.7520,0.7754,-0.1538,0.6857,0.0740,0.0136,0.9766
5,DisparateImpactRemover (repair=1.0),pre-processing,0.7505,0.7734,-0.1414,0.7059,0.0774,0.0012,0.7503
6,"LFR (k=10, Az=10.0, balanced)",pre-processing,0.7280,0.7527,-0.0372,0.9123,0.0562,0.1030,6.8657
7,"LFR (k=5, Az=50.0, fairness-dominant)",pre-processing,0.4700,0.5279,-0.0165,0.9820,0.0110,0.1237,4.5345
8,PrejudiceRemover (eta=1.0),in-processing,0.7502,0.7727,-0.2483,0.5419,0.1699,0.1081,18.9526
9,PrejudiceRemover (eta=5.0),in-processing,0.7389,0.7686,-0.1340,0.7437,0.0900,0.0062,24.0273


In [15]:
# --- Figure: tradeoff, with the degenerate methods marked -----------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

STAGE_STYLE = {
    "none": ("black", "*", 260),
    "pre-processing": ("#8172B2", "o", 90),
    "in-processing": ("#C44E52", "s", 90),
    "post-processing": ("#55A868", "^", 90),
}

# A method that approves (almost) everyone or (almost) nobody IN SOME GROUP is
# degenerate, however good its fairness number looks. The test has to be per
# group: approving 52% of men and 0% of women averages to a healthy 26%.
group_rates = mitigation_results[["selection_rate_Male", "selection_rate_Female"]]
degenerate = (group_rates.min(axis=1) < 0.02) | (group_rates.max(axis=1) > 0.98)

for stage, (colour, marker, size) in STAGE_STYLE.items():
    mask = (mitigation_results.stage == stage) & ~degenerate
    axes[0].scatter(mitigation_results.loc[mask, "DP_difference"],
                    mitigation_results.loc[mask, "accuracy_vs_observed"],
                    c=colour, marker=marker, s=size, label=stage,
                    edgecolors="white", linewidths=0.8, zorder=3)

axes[0].scatter(mitigation_results.loc[degenerate, "DP_difference"],
                mitigation_results.loc[degenerate, "accuracy_vs_observed"],
                facecolors="none", edgecolors="red", marker="X", s=170,
                linewidths=2, label="DEGENERATE (constant classifier)", zorder=4)
axes[0].set_xlabel("|demographic parity difference|")
axes[0].set_ylabel("accuracy vs observed labels")
axes[0].set_title("A good fairness number is not enough:\nthe red crosses are constant classifiers")
axes[0].legend(fontsize=8)

# Selection rate per group makes the failure modes unmistakable.
ordered = mitigation_results.sort_values("selection_rate_Female")
positions = np.arange(len(ordered))
axes[1].barh(positions - 0.2, ordered["selection_rate_Male"], 0.4,
             label="Male", color=common.PRIVILEGED_COLOR)
axes[1].barh(positions + 0.2, ordered["selection_rate_Female"], 0.4,
             label="Female", color=common.UNPRIVILEGED_COLOR)
axes[1].set_yticks(positions)
axes[1].set_yticklabels(ordered["method"], fontsize=6.5)
axes[1].set_xlabel("selection rate")
axes[1].set_title("Per-group approval rates")
axes[1].legend(fontsize=8)

fig.tight_layout()
common.savefig(fig, common.AIF360_FIGURES, "aif_fig5_mitigation_tradeoff.png")
plt.show()

print(f"{int(degenerate.sum())} of {len(mitigation_results)} methods left some group")
print("approved almost always or almost never:")
for _, row in mitigation_results[degenerate].iterrows():
    print(f"  {row['method']:44s} Male={row['selection_rate_Male']:.3f} "
          f"Female={row['selection_rate_Female']:.3f}")

3 of 23 methods left some group
approved almost always or almost never:
  GerryFairClassifier (FP, gamma=0.01)         Male=1.000 Female=1.000
  CalibratedEqOdds (fnr)                       Male=0.996 Female=0.312
  CalibratedEqOdds (fpr)                       Male=0.521 Female=0.000


In [16]:
# --- Figure: the CalibratedEqOdds lesson ----------------------------------
fig, ax = plt.subplots(figsize=(10, 4.2))

# `calibrated` carries the score-based rates, computed on the datasets that
# still had scores attached.
subset = calibrated.copy()
subset["label"] = subset["method"].str.replace("CalibratedEqOdds ", "cost=", regex=False)

metrics = [
    ("generalized_FPR_difference", "#4C72B0"),
    ("generalized_FNR_difference", "#DD8452"),
    ("selection_rate_Female", "#C44E52"),
]
positions = np.arange(len(subset))
width = 0.26

for offset, (metric, colour) in enumerate(metrics):
    values = subset[metric].astype(float)
    bar_positions = positions + (offset - 1) * width
    ax.bar(bar_positions, values, width, label=metric.replace("_", " "), color=colour)
    # Label every bar. A bar of height zero is invisible otherwise, and the
    # zero bars are the whole point of this figure.
    for x, value in zip(bar_positions, values):
        ax.text(x, value + (0.012 if value >= 0 else -0.028), f"{value:.3f}",
                ha="center", fontsize=7, fontweight="bold")

ax.axhline(0, c="black", lw=1)
ax.set_ylim(subset[[m for m, _ in metrics]].astype(float).values.min() - 0.06,
            subset[[m for m, _ in metrics]].astype(float).values.max() + 0.07)
ax.set_xticks(positions)
ax.set_xticklabels(subset["label"], fontsize=8)
ax.set_ylabel("value")
ax.set_title("Each cost_constraint equalises its own target and wrecks the others\n"
             "(note the female selection rate under cost=fpr)", fontsize=11)
ax.legend(fontsize=8)

fig.tight_layout()
common.savefig(fig, common.AIF360_FIGURES, "aif_fig6_calibrated_eqodds.png")
plt.show()

## What this notebook establishes

**AIF360's functional profile for mitigation:**

| strength | detail |
|----------|--------|
| Pre-processing depth | 3 working algorithms against Fairlearn's 2, spanning genuinely different mechanisms (reweighting, distributional repair, representation learning) |
| `Reweighing` | free, interpretable, estimator-agnostic, and Fairlearn has nothing like it |
| Post-processing depth | 3 algorithms with tunable targets against Fairlearn's 1 |
| Cost-constraint control | `CalibratedEqOdds` lets you name which error type to equalise |
| Rich subgroup fairness | `GerryFairClassifier` targets exponentially many subgroups - in principle |
| Interoperability | `ExponentiatedGradientReduction` wraps Fairlearn, so nothing is lost by choosing AIF360 |

| limitation | detail |
|------------|--------|
| In-processing is not estimator-agnostic | `PrejudiceRemover` and `MetaFairClassifier` are welded to their own model classes; only the Fairlearn wrapper is general |
| Maintenance gaps | `MetaFairClassifier` raises `TypeError` for every tau that requests fairness |
| Fragile defaults | `GerryFairClassifier` collapses to a constant classifier; `LFR` needs hand-tuned weights to avoid the same |
| Silent calibration requirement | post-processors need a held-out split and warn you if you forget |
| Heavier API | every algorithm speaks `BinaryLabelDataset`, so encoding and unwrapping bracket every call |

### The best method on this dataset

`RejectOptionClassification` targeting statistical parity is the standout. It
cut the selection-rate gap from -0.204 to -0.031 **and** scored higher against
the oracle fair labels (0.783) than the unmitigated baseline did (0.777) - the
only method that improved on both axes at once.

The mechanism explains why: rather than shifting a threshold wholesale, it only
flips decisions inside a band of uncertainty around the boundary, favouring the
unprivileged group there. Confident decisions are left untouched. Since
mechanism B1 flipped *qualified* applicants to denied, many of them sit near
the boundary - exactly where this algorithm looks.

That is a good reminder that the best algorithm depends on the shape of the
bias, not on the stage it operates at.

### The three findings that generalise beyond this dataset

**1. Breadth is real, but so is the maintenance gap.** AIF360 offers roughly
twice the algorithms. On this dataset one is broken outright, one is degenerate
under its documented defaults, and one silently needs a data split the API
never mentions. Fairlearn's smaller catalogue all worked first time.

**2. The two libraries are not really competitors at in-processing.** AIF360's
own reduction algorithm *is* Fairlearn, called through an adapter, and it
reproduces Fairlearn's predictions exactly. The genuine differences are at the
pre- and post-processing stages.

**3. Optimising one fairness metric can destroy every other one, silently.**
`CalibratedEqOdds(fpr)` equalised its target almost perfectly by approving zero
women. Its chosen metric improved. Accuracy barely moved. Only a panel of
metrics - selection rates per group, parity, odds, and the degeneracy check -
catches it. This is the concrete argument for the detection notebooks: you
cannot mitigate against a single number.